# ML-05: Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

ML-05 was retired on 2026-07-13; its core leakage audit now lives in ML-04 ([`w03_data_contract.ipynb`](w03_data_contract.ipynb)). This notebook is a summary pointing at `w03_data_contract.ipynb` and [`notebooks/03_working_with_the_full_release.ipynb`](../../notebooks/03_working_with_the_full_release.ipynb) rather than an independent run. It documents the retained five-feature domain vector, the deliberate `future30_impressions` leakage experiment (honest AP 0.750 vs leaky AP 0.998), exclusions, and field availability.


## 1. Build the feature vector

The five retained inputs are documented and built in the investigation notebook: `log_recent30_impressions`, `recent30_ctr_pct`, `recent30_avg_position`, `recent30_active_days`, and `content_age_days`. The feature frame uses the recent 30 days before the March 31 cutoff and excludes future April values.

In [5]:
DOMAIN_FEATURES = [
    'log_recent30_impressions',
    'recent30_ctr_pct',
    'recent30_avg_position',
    'recent30_active_days',
    'content_age_days',
 ]

print('Investigation feature vector:', DOMAIN_FEATURES)
assert len(DOMAIN_FEATURES) == 5

Investigation feature vector: ['log_recent30_impressions', 'recent30_ctr_pct', 'recent30_avg_position', 'recent30_active_days', 'content_age_days']


## 2. Feature notes (meaning, missing, categorical, available-when?)

The investigation documents availability at the decision moment: recent GSC aggregates are available through March 31; `content_age_days` comes from page metadata; `0` position values are treated as missing position data before aggregation. Client and content identifiers are context only. The investigation also applies `gsc_data_available IS TRUE` rather than treating unavailable data as zero activity.

In [6]:
feature_notes = {
    'log_recent30_impressions': 'numeric engineered visibility; log-scaled recent GSC impressions',
    'recent30_ctr_pct': 'numeric engineered click efficiency; recent clicks / impressions',
    'recent30_avg_position': 'numeric engineered position; zero sentinel treated as missing',
    'recent30_active_days': 'numeric engineered consistency; days with impressions',
    'content_age_days': 'numeric metadata feature; age at the cutoff',
}

for feature, note in feature_notes.items():
    print(f'{feature}: {note}; available before March 31 scoring.')

log_recent30_impressions: numeric engineered visibility; log-scaled recent GSC impressions; available before March 31 scoring.
recent30_ctr_pct: numeric engineered click efficiency; recent clicks / impressions; available before March 31 scoring.
recent30_avg_position: numeric engineered position; zero sentinel treated as missing; available before March 31 scoring.
recent30_active_days: numeric engineered consistency; days with impressions; available before March 31 scoring.
content_age_days: numeric metadata feature; age at the cutoff; available before March 31 scoring.


## 3. The leakage hunt

The investigation intentionally added `future30_impressions`, which is measured from April 1–30, to the model. Average precision rose from `0.750` for the honest five-feature model to `0.998` for the leaky version. That jump is the warning: the future value directly reveals the outcome and must be deleted before scoring.

In [7]:
LEAKY_FEATURES = DOMAIN_FEATURES + ['future30_impressions']
HONEST_FEATURES = [feature for feature in LEAKY_FEATURES if feature != 'future30_impressions']

print('Deliberate leakage feature:', 'future30_impressions')
print('Honest feature count after removal:', len(HONEST_FEATURES))
assert 'future30_impressions' not in HONEST_FEATURES
assert len(HONEST_FEATURES) == 5

Deliberate leakage feature: future30_impressions
Honest feature count after removal: 5


## 4. What I excluded and why

- `future30_impressions`: future-window information and directly label-related.
- `is_declining_next30`: the target itself, never a feature.
- `client_hash_id`, `content_hash_id`: pseudonymous context identifiers for grouping and joins only.
- `report_date` and `month`: time controls for window construction, not page features.
- `gsc_data_available` and `ga4_data_available`: availability controls, not page-quality signals.
- `provider_used` and `model_used`: production metadata, not content-performance evidence.
- Fixed-window query signals: excluded until their window is aligned so it cannot overlap the future label.

In [8]:
excluded_fields = {
    'future30_impressions': 'future outcome leakage',
    'is_declining_next30': 'label, not a feature',
    'client_hash_id': 'grouping and split context',
    'content_hash_id': 'join and inspection context',
    'provider_used': 'production metadata',
    'model_used': 'production metadata',
}

for field, reason in excluded_fields.items():
    print(f'{field}: {reason}')

print('See the executed investigation notebook for the full model comparison and selection audit.')

future30_impressions: future outcome leakage
is_declining_next30: label, not a feature
client_hash_id: grouping and split context
content_hash_id: join and inspection context
provider_used: production metadata
model_used: production metadata
See the executed investigation notebook for the full model comparison and selection audit.


## Self-check

- [x] Required ML-04 contract remains in [`w03_data_contract.ipynb`](w03_data_contract.ipynb)
- [x] Full feature construction and leakage comparison are linked in [`03_working_with_the_full_release.ipynb`](../../notebooks/03_working_with_the_full_release.ipynb)
- [x] The deliberate future-window feature is removed from the honest feature list
- [x] Exclusions and availability rules are documented
- [x] Notebook was run in the target environment; this optional companion is ready to commit